# Day 11 — Practice Session · **SOLUTIONS**
### Exceptions & Modules · Python for Data Science

**Prepared by Srinivasa Sai Chava**  ·  Boston University

---

> **Instructor copy.** Every question is followed by the answer and the reasoning.
> The student copy (`Day11_Practice_Questions.ipynb`) is identical minus the answer blocks.

| Part | Focus | Questions |
|---|---|---|
| A | Predict the output | 8 |
| B | Spot & fix the bug | 4 |
| C | Write the code | 5 |
| D | Challenge | 2 |

---
# Part A — Predict the Output  *(8 min)*

---
# Part A — Predict the Output

**Teaching note:** A3 (finally with a return) and A6 (except ordering) are the two that
catch people out. A8 is the `import *` shadowing demo — run it live if there is time.

### A1. What prints?

In [ ]:
try:
    x = int("abc")
except ValueError:
    print("caught it")
    x = 0
print(x)

*Your prediction:*  

> ### ✅ Answer A1
> ```
> caught it
> 0
> ```
> **Why:** `int("abc")` raises `ValueError`, so the `try` block stops there and the `except`
> block runs instead. The assignment `x = int("abc")` never completed, which is why `x = 0`
> inside the handler matters — without it, the `print(x)` would raise `NameError`.

### A2. Which lines print?

In [ ]:
try:
    print("A")
    result = 10 / 2
except ZeroDivisionError:
    print("B")
else:
    print("C")
finally:
    print("D")

*Your prediction:*  

> ### ✅ Answer A2
> ```
> A
> C
> D
> ```
> **Why:** nothing failed, so `except` is skipped and `else` runs. `finally` always runs.
>
> If the divisor had been `0` you would get `A`, `B`, `D` — the `else` skipped and the
> `except` run instead.

### A3. What does this function return, and what prints?

In [ ]:
def f():
    try:
        return "from try"
    finally:
        print("finally ran")

print(f())

*Your prediction:*  

> ### ✅ Answer A3
> ```
> finally ran
> from try
> ```
> **Why:** `finally` runs **even when the try block returns** — and it runs *before* the
> value is handed back, which is why "finally ran" prints first.
>
> That guarantee is exactly what makes `finally` the right place for cleanup, and it is the
> mechanism Day 9's `with open(...)` is built on.

### A4. What is printed?

In [ ]:
try:
    d = {"a": 1}
    print(d["z"])
except LookupError:
    print("lookup failed")

*Your prediction:*  

> ### ✅ Answer A4
> ```
> lookup failed
> ```
> **Why:** `KeyError` inherits from `LookupError`, so catching the parent catches the child.
>
> `IndexError` also inherits from `LookupError`, so the same handler would catch
> `[1, 2][99]`. This is Day 8's inheritance deciding which `except` block matches —
> `except` is really an `isinstance` check.

### A5. What prints?

In [ ]:
def check(age):
    if age < 0:
        raise ValueError(f"negative: {age}")
    return age

try:
    check(-5)
    print("no problem")
except ValueError as e:
    print("caught:", e)

*Your prediction:*  

> ### ✅ Answer A5
> ```
> caught: negative: -5
> ```
> **Why:** `raise` stops the function immediately, so `print("no problem")` is never
> reached. `as e` binds the exception object, and printing it shows the message you passed in.
>
> Note the message says **what** was wrong and **what value** caused it. "Invalid input"
> would have told the caller nothing.

### A6. Which block runs?

In [ ]:
try:
    int("abc")
except Exception:
    print("general")
except ValueError:
    print("specific")

*Your prediction:*  

> ### ✅ Answer A6
> ```
> general
> ```
> **Why:** Python tries each `except` block in order and takes the **first** that matches.
> `ValueError` inherits from `Exception`, so the general block matches first and the
> specific one is unreachable.
>
> **This is the elif-ordering rule from Day 2.** Put specific types first and the general
> one last. Some linters will warn about the unreachable block; Python itself will not.

### A7. Does this work?

In [ ]:
class MyError:
    pass

raise MyError("something went wrong")

*Your prediction:*  

> ### ✅ Answer A7
> ```
> TypeError: exceptions must derive from BaseException
> ```
> **Why:** only classes inheriting from `BaseException` can be raised. A plain class is not
> an exception, however error-like its name.
>
> **The fix is one word:**
> ```python
> class MyError(Exception):
>     pass
> ```

### A8. What does the second print show?

In [ ]:
def sqrt(x):
    return "my own sqrt"

print(sqrt(9))

from math import *

print(sqrt(9))

*Your prediction:*  

> ### ✅ Answer A8
> ```
> my own sqrt
> 3.0
> ```
> **Why:** `from math import *` dumps every name from `math` into your namespace, silently
> replacing your own `sqrt` function. No warning, no error — your code simply stops doing
> what you wrote.
>
> **This is why `import *` is banned in essentially every professional codebase.** Use
> `import math` and call `math.sqrt()`, so the source of every name stays visible.

---
# Part B — Spot & Fix the Bug  *(7 min)*

---
# Part B — Spot & Fix the Bug

**Teaching note:** B1 and B2 are the two anti-patterns from the session. B2 is the more
dangerous — it produces a wrong answer rather than an error.

### B1. This hides a typo. Find it.

In [ ]:
def process(row):
    return row["mark"] * 2

try:
    print(process({"marks": 88}))
except:
    print("something went wrong")

*What's wrong:* 

*Your fix:*

> ### ✅ Answer B1
> **Symptom:** prints "something went wrong" and you have no idea why.
> **Cause:** the dictionary key is `"marks"` but the function reads `"mark"` — a typo that
> raises `KeyError`. The bare `except:` swallows it along with everything else, including
> `NameError`s, your own typos, and `KeyboardInterrupt`.

In [ ]:
def process(row):
    return row["mark"] * 2

try:
    print(process({"marks": 88}))
except KeyError as e:
    print(f"missing column: {e}")     # now you can SEE the problem

# Naming the exception turns an invisible bug into a readable message.
# If you genuinely need a catch-all, use  except Exception  - at least that
# leaves KeyboardInterrupt alone so the program can still be stopped.

### B2. The total is wrong and nobody notices.

In [ ]:
rows = [{"mark": "88"}, {"mark": "abc"}, {"mark": "71"}]

total = 0
for row in rows:
    try:
        total += int(row["mark"])
    except ValueError:
        pass

print("total:", total)

*What's wrong:* 

*Your fix:*

> ### ✅ Answer B2
> **Symptom:** prints `total: 159`. No error — but one row was silently discarded and
> nothing says so.
> **Cause:** `except ValueError: pass` hides the problem completely. Six months later
> someone notices the numbers are wrong and there is no trace of why.

In [ ]:
rows = [{"mark": "88"}, {"mark": "abc"}, {"mark": "71"}]

total, skipped = 0, []
for n, row in enumerate(rows, start=1):
    try:
        total += int(row["mark"])
    except ValueError:
        skipped.append(f"row {n}: {row['mark']!r}")   # counted, not hidden

print("total  :", total)
print("skipped:", skipped)

# If you deliberately ignore something, SAY SO. Count it, log it, or return it.
# Silence is the enemy of debugging.

### B3. The specific handler never runs.

In [ ]:
def load(path):
    try:
        with open(path, encoding="utf-8") as f:
            return f.read()
    except Exception:
        return "could not read the file"
    except FileNotFoundError:
        return "no such file"

print(load("nope.txt"))

*What's wrong:* 

*Your fix:*

> ### ✅ Answer B3
> **Symptom:** always returns the vague message, never the specific one.
> **Cause:** `except Exception` comes first and matches everything, so the
> `FileNotFoundError` block is unreachable. You also lose the ability to tell a missing
> file apart from a permissions problem or a bad encoding.

In [ ]:
def load(path):
    try:
        with open(path, encoding="utf-8") as f:
            return f.read()
    except FileNotFoundError:          # specific FIRST
        return "no such file"
    except PermissionError:
        return "not allowed to read it"
    except OSError as e:               # general LAST, and still named
        return f"could not read the file: {e}"

print(load("nope.txt"))

# Order specific to general, exactly like an elif chain.

### B4. This function reports an error that nobody can act on.

In [ ]:
def set_mark(mark):
    if not 0 <= mark <= 100:
        return "invalid"
    return mark

marks = [set_mark(88), set_mark(150)]
print(sum(marks))

*What's wrong:* 

*Your fix:*

> ### ✅ Answer B4
> **Symptom:** `TypeError: unsupported operand type(s) for +: 'int' and 'str'` — raised by
> `sum()`, far away from the function that actually had the problem.
> **Cause:** returning an error message mixes error reporting into the normal return value.
> The caller has to remember to check, and if they forget, the failure surfaces somewhere
> unrelated with a confusing message.

In [ ]:
class InvalidMarkError(ValueError):
    """A mark outside the range 0-100."""


def set_mark(mark):
    if not 0 <= mark <= 100:
        raise InvalidMarkError(f"expected 0-100, got {mark}")
    return mark


# Now the problem is reported where it happens, with a usable message
for m in [88, 150]:
    try:
        print("accepted:", set_mark(m))
    except InvalidMarkError as e:
        print("rejected:", e)

# Inheriting from ValueError means code that already catches ValueError
# still works, while code that wants only THIS error can ask for it by name.

---
# Part C — Write the Code  *(12 min)*

---
# Part C — Write the Code

**Teaching note:** C2 is the retry loop they will reuse constantly. C5 is the module
exercise — make sure everyone actually runs it, since writing a file from a notebook is new.

### C1. Safe division
Write `safe_divide(a, b)` that returns `a / b`, but returns `None` and prints a message
if `b` is zero. Test it with `10 / 2` and `10 / 0`.

In [ ]:
# your code here

In [ ]:
def safe_divide(a, b):
    """Return a / b, or None if b is zero."""
    try:
        return a / b
    except ZeroDivisionError:
        print(f"cannot divide {a} by zero")
        return None


print(safe_divide(10, 2))     # 5.0
print(safe_divide(10, 0))     # message, then None

# You could also check  if b == 0  first. try/except is preferred when the
# failure is genuinely exceptional; an if-check is fine when it is expected.

### C2. Keep asking until the input is valid
Write `get_mark()` that repeatedly asks the user for a mark until they enter a whole number
between 0 and 100, then returns it.

Handle a non-numeric entry and an out-of-range number with **different** messages.

In [ ]:
# your code here

In [ ]:
def get_mark():
    """Ask until the user gives a whole number between 0 and 100."""
    while True:                                   # Day 2's while + break
        raw = input("Enter a mark (0-100): ")
        try:
            mark = int(raw)
        except ValueError:
            print("  that is not a whole number")
            continue                              # ask again
        if not 0 <= mark <= 100:
            print("  out of range")
            continue
        return mark                               # valid - leave the loop


# mark = get_mark()
# print("recorded:", mark)
print("Uncomment the two lines above to try it interactively.")

# The try block wraps ONLY int(raw) - the line that can fail.
# The range check sits outside it, because being out of range is not an
# exception, it is an ordinary condition.

### C3. A custom exception
Write an exception class `NegativeAgeError` and a function `set_age(age)` that raises it for
a negative age. Catch it and print the message.

In [ ]:
# your code here

In [ ]:
class NegativeAgeError(ValueError):
    """An age below zero."""


def set_age(age):
    """Store an age, rejecting negatives."""
    if age < 0:
        raise NegativeAgeError(f"age cannot be negative, got {age}")
    return age


for a in [23, -5]:
    try:
        print("accepted:", set_age(a))
    except NegativeAgeError as e:
        print("rejected:", e)

print()
print("is it a ValueError too?", issubclass(NegativeAgeError, ValueError))
# Inheriting from ValueError rather than Exception is a deliberate choice:
# a negative age IS a bad value, so existing code catching ValueError still works.

### C4. Use the standard library
Without writing any loops of your own, use `statistics` and `collections.Counter` to print:

1. the mean and median of the marks below
2. the most common mark

In [ ]:
marks = [88, 71, 64, 88, 90, 71, 88]

# your code here

In [ ]:
import statistics
from collections import Counter

marks = [88, 71, 64, 88, 90, 71, 88]

print("mean       :", round(statistics.mean(marks), 2))
print("median     :", statistics.median(marks))
print("most common:", Counter(marks).most_common(1))
print("all counts :", Counter(marks))

# mean: 80   median: 88   most common: [(88, 3)]
#
# Note statistics.mean returns an int (80) rather than 80.0 here, because
# the seven marks divide evenly. It returns a float when they do not.
#
# Both of these are in the standard library - already installed and better
# tested than anything you would write. statistics is a preview of Module 2.

### C5. Write and import your own module
Create a file `mytools.py` containing a function `shout(text)` that returns the text
uppercased with an exclamation mark. Then import it and use it.

In [ ]:
# your code here

In [ ]:
# Write the module to disk
lines = [
    'def shout(text):',
    '    """Return the text uppercased with an exclamation mark."""',
    '    return text.upper() + "!"',
    '',
    'if __name__ == "__main__":',
    '    print(shout("run directly"))',
]

with open("mytools.py", "w", encoding="utf-8") as f:
    f.write("\n".join(lines))

# Now import and use it
import mytools

print(mytools.shout("hello"))          # HELLO!
print(mytools.shout.__doc__)

# Notice the __main__ block did NOT print - that code runs only when the file
# is executed directly, not when it is imported. That is how one file can be
# both a reusable module and a runnable script.

---
# Part D — Challenge  *(3 min, or take home)*

---
# Part D — Challenge

**Teaching note:** D1 is the session in one function — expected failures handled, unexpected
ones raised. D2 rewards anyone who understood the re-raise slide.

### D1. A loader that survives bad data
Write `load_marks(path)` that reads a CSV with `name` and `mark` columns and returns
**two** lists: the good rows, and a description of each bad one.

- A mark that is not a number, or outside 0–100, makes the row bad — but the load continues.
- A missing **file** is not something this function should hide.

In [ ]:
import csv

# Build a test file with two deliberately bad rows
with open("test.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["name", "mark"])
    w.writerows([["Ravi", 88], ["Sara", "abc"], ["Amit", 150], ["Neha", 71]])

# your code here

In [ ]:
import csv


class InvalidMarkError(ValueError):
    """A mark that is not a number between 0 and 100."""


def parse_mark(raw):
    try:
        mark = int(raw)
    except ValueError:
        raise InvalidMarkError(f"not a number: {raw!r}")
    if not 0 <= mark <= 100:
        raise InvalidMarkError(f"out of range: {mark}")
    return mark


def load_marks(path):
    """Return (good_rows, bad_row_descriptions). Missing files are NOT caught."""
    good, bad = [], []
    with open(path, newline="", encoding="utf-8") as f:
        for n, row in enumerate(csv.DictReader(f), start=2):   # 2 = first data line
            try:
                row["mark"] = parse_mark(row["mark"])
                good.append(row)
            except InvalidMarkError as e:
                bad.append(f"line {n}: {e}")
    return good, bad


with open("test.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["name", "mark"])
    w.writerows([["Ravi", 88], ["Sara", "abc"], ["Amit", 150], ["Neha", 71]])

good, bad = load_marks("test.csv")
print("good:", [(r["name"], r["mark"]) for r in good])
print("bad :")
for b in bad:
    print("   ", b)

# A missing file is NOT caught - it goes up to the caller
try:
    load_marks("no_such_file.csv")
except FileNotFoundError:
    print("\nmissing file propagated, as it should")

# THE PRINCIPLE: a bad row is EXPECTED, so it is handled.
# A missing file is NOT expected, so the caller decides what to do.

### D2. Log it, then let it through
Write `risky_load(path)` that:

- logs a message when the file is missing
- but still lets the `FileNotFoundError` reach the caller

Prove it works by catching the error outside the function.

In [ ]:
# your code here

In [ ]:
def risky_load(path):
    """Read a file, recording any failure but not swallowing it."""
    try:
        with open(path, encoding="utf-8") as f:
            return f.read()
    except FileNotFoundError:
        print(f"  [log] could not find {path}")
        raise                      # bare raise re-throws the SAME exception


try:
    risky_load("nope.txt")
except FileNotFoundError as e:
    print("caller handled it:", type(e).__name__)

# A bare  raise  inside an except block re-throws what you just caught,
# keeping the original traceback intact.
#
# Use it when you want to RECORD that something happened but still let the
# caller decide what to do about it.
#
# LOGGING IS NOT HANDLING. Printing the error does not mean you dealt with it.

---
## Done? Self-check

- [ ] I read a traceback from the bottom up
- [ ] I can say why `except:` is worse than `except Exception:`
- [ ] I know why `except: pass` is dangerous
- [ ] I know why a general `except` written first makes later ones unreachable
- [ ] I know when `else` runs and when `finally` runs
- [ ] I can write a custom exception, and say why it inherits from `Exception`
- [ ] I know why `from x import *` is never the right choice

### Homework
1. Add proper exception handling to your Day 9 CSV functions.
2. Write a custom exception for one rule in your own code.
3. Split a notebook into a `helpers.py` module and import it.

### Next class — Topic 1.11: NumPy
Array creation and indexing: `np.array`, `arange`, `zeros` and `ones`, slicing, reshaping.

---
*Slides & notebooks by Srinivasa Sai Chava · Boston University*

---
## Wrap-up — running the last 5 minutes

Three cold-call questions:

1. *"Where do you start reading a traceback?"* → the bottom line, which names the error.
2. *"What is wrong with `except: pass`?"* → the bug becomes invisible. Count it or log it.
3. *"Why put specific `except` blocks before general ones?"* → the first match wins, so a
   general one written first makes the rest unreachable.

**Common misconceptions to watch for today**

| Misconception | Correction |
|---|---|
| "Read the traceback from the top" | The bottom line names the error; read upwards only to trace the call |
| "`except:` is a safe default" | It catches typos and Ctrl+C; use a named exception |
| "Catching an error means handling it" | `pass` hides it. Count it, log it, or re-raise |
| "Order of `except` blocks does not matter" | First match wins — specific first, general last |
| "`finally` is skipped on `return`" | It always runs, even before the value is handed back |
| "Returning an error message is fine" | It fails later, far from the cause. `raise` instead |
| "Any class can be raised" | Only classes inheriting from `BaseException` |
| "`import *` saves typing" | It silently shadows your own names |

**The distinction worth repeating:** an exception you did **not** expect should stop the
program loudly. An exception you **did** expect should be handled quietly. Nearly all bad
error handling comes from confusing the two.

**Homework given:** exception handling on the Day 9 CSV functions; one custom exception;
split a notebook into a module.

**Next session:** Topic 1.11 — NumPy. This closes the pure-Python half of Module 1. From
tomorrow every session begins with an import, and the loops from Day 2 start disappearing
into vectorised operations.